# Analytics Overview

**Le notebook présent sert à observer différentes métriques et visuels sur nos données**

    > à exécuter uniquement depuis Databricks ou Google Collab

In [0]:
# Dans une cellule, vérifie que le fichier est accessible
dbutils.fs.ls("/Volumes/main/default/raw/")

In [0]:
import os
import sys
os.getcwd()
import os
os.chdir("/Workspace/Users/karl.sondeji@aivancity.education/Spark-pipeline-on-Online-Retail")
os.getcwd()

**Exécution du pipeline**

In [0]:
from src.main import run_pipeline
run_pipeline()

**Importation des fonctions**

In [0]:
import matplotlib.pyplot as plt

from src.analytics.temporal_analysis import get_monthly_revenue
from src.analytics.customer_analysis import get_rfm_segment_summary, get_cohort_retention
from src.analytics.returns_analysis import get_return_rate_by_category
from src.analytics.pareto_analysis import get_customer_pareto, get_top_products


In [0]:
gold = "/Volumes/main/default/raw/gold"

In [0]:
bronze = "/Volumes/main/default/raw/bronze"

## Analyse temporelle

In [0]:
# --- Tendance mensuelle ---
monthly_pd = get_monthly_revenue(spark, gold).toPandas()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_pd["year_month"], monthly_pd["total_revenue"], marker="o")
ax.set_title("Évolution du CA mensuel en millions")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

Le mois de novembre 2011 est celui avec le plus gros pic avec près de 1.2M de CA réalisé.

In [0]:
# --- Segments RFM (Recency / Frequency / Monetary) ---
rfm_pd = get_rfm_segment_summary(spark, gold).toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rfm_pd["rfm_segment"], rfm_pd["total_revenue"])
ax.set_title("CA par segment RFM")
plt.tight_layout()
plt.show()

Près de 80% du CA provient des clients de la catégorie champion, à savoir les client avec un score RFM supérieur à 10, ce sont les clients qui ont acheté récemment, qui achètent souvent et qui dépensent beaucoup au total.

In [0]:
# --- Pareto clients ---
pareto_pd = get_customer_pareto(spark, gold).toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pareto_pd["rank"], pareto_pd["cumulative_pct"])
ax.axhline(80, color="red", linestyle="--", label="80% du CA")
ax.set_title("Courbe de Pareto — clients")
ax.set_xlabel("Nombre de clients (triés par CA décroissant)")
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT
  COUNT(*) AS nb_lignes,
  COUNT(DISTINCT CustomerID) AS nb_clients,
  COUNT(DISTINCT InvoiceNo) AS nb_factures
FROM delta.`/Volumes/main/default/raw/gold`;

Sur le graphique ci-dessus, on peut voir que les 1200 meilleurs clients sont responsable de 80% du CA sur près de 4312 clients.

In [0]:
# --- Taux de retour par catégorie (retours = bronze, ventes = gold) ---
bronze = "/Volumes/main/default/raw/bronze"
returns_pd = get_return_rate_by_category(spark, gold, bronze_path=bronze).toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(returns_pd["product_category"], returns_pd["return_rate_pct"])
ax.set_xlabel("Taux de retour (%)")
ax.set_title("Taux de retour par catégorie")
plt.tight_layout()
plt.show()

Le taux de retour moyen par catégorie est d'environ 2%, la catégorie Cakes a le taux de retours le plus élevé avec près de 3.8%.

In [0]:
# --- Top produits (tables) ---
top_revenue, top_volume = get_top_products(spark, gold, n=10)
print("Top 10 produits par CA:")
display(top_revenue)
print("Top 10 produits par volume:")
display(top_volume)

**Evolution de la qualité des données**

In [0]:
import plotly.graph_objects as go

In [0]:
def get_row_counts_by_phase(spark, raw_path, bronze_path, silver_path, gold_path):
    """Compte les lignes à chaque étape du pipeline."""
    raw_count = spark.read.option("header", True).csv(raw_path).count()
    bronze_count = spark.read.format("delta").load(bronze_path).count()
    silver_count = spark.read.format("delta").load(silver_path).count()
    gold_count = spark.read.format("delta").load(gold_path).count()
    return {
        "Raw (CSV)": raw_count,
        "Bronze": bronze_count,
        "Silver (nettoyé)": silver_count,
        "Gold (agrégeable)": gold_count,
    }

counts = get_row_counts_by_phase(
    spark,
    "/Volumes/main/default/raw/Online_Retail.csv",
    "/Volumes/main/default/raw/bronze",
    "/Volumes/main/default/raw/silver",
    "/Volumes/main/default/raw/gold",
)

fig = go.Figure(go.Funnel(
    y=list(counts.keys()),
    x=list(counts.values()),
    textinfo="value+percent initial",
))
fig.update_layout(title="Volumétrie du pipeline : lignes conservées par phase")
fig.show()

In [0]:
# Lignes perdues entre silver et Bronze
loss_bronze_to_silver = counts["Bronze"] - counts["Silver (nettoyé)"]
print(f"Lignes supprimées au nettoyage silver : {loss_bronze_to_silver} "
      f"({loss_bronze_to_silver/counts['Bronze']*100:.1f}%)")

Complément utile : pourquoi on perd des lignes à chaque étape
Les différentes phases de nettoyage ont conduit à la suppression de près d'un tiers des lignes.
Ceci s'explique par la suppression des doublons, des commandes abandonnées, des lignes corrompues etc.

In [0]:
from pyspark.sql import functions as F
from src.quality.rejection_breakdown import get_rejection_breakdown

In [0]:
# import importlib, sys
# for k in list(sys.modules):
#     if k.startswith("src."):
#         del sys.modules[k]

In [0]:
get_rejection_breakdown(spark, "/Volumes/main/default/raw/bronze")

In [0]:
rejections = get_rejection_breakdown(spark, "/Volumes/main/default/raw/bronze")

fig = go.Figure(go.Bar(x=list(rejections.keys()), y=list(rejections.values())))
fig.update_layout(title="Lignes concernées par chaque règle de qualité (non-exclusif)")
fig.show()

**Comparaison de versions Delta**

In [0]:
def get_version_delta_pd(spark, table_name, v0=0, v1=1):
    df_v0 = spark.sql(f"SELECT * FROM {table_name} VERSION AS OF {v0}")
    df_v1 = spark.sql(f"SELECT * FROM {table_name} VERSION AS OF {v1}")

    rev_v0 = df_v0.groupBy("Country").agg(F.sum(F.col("Quantity")*F.col("UnitPrice")).alias("revenue_v0"))
    rev_v1 = df_v1.groupBy("Country").agg(F.sum(F.col("Quantity")*F.col("UnitPrice")).alias("revenue_v1"))

    delta = (
        rev_v0.join(rev_v1, "Country", "outer")
              .fillna(0)
              .withColumn("delta", F.round(F.col("revenue_v1") - F.col("revenue_v0"), 2))
              .filter(F.col("delta") != 0)
              .orderBy(F.desc("delta"))
    )
    return delta.toPandas()

delta_pd = get_version_delta_pd(spark, "main.default.phase4")

colors = ["green" if x >= 0 else "red" for x in delta_pd["delta"]]
fig = go.Figure(go.Bar(x=delta_pd["Country"], y=delta_pd["delta"], marker_color=colors))
fig.update_layout(title="Écart de CA par pays entre versions Delta (v0 → v1)")
fig.show()

Presque qu'aucune différence entre les versions,le MERGE ne change presque rien, cela montre qu'il est cohérent.

In [0]:
def get_delta_operations_summary(spark, table_name):
    history = spark.sql(f"DESCRIBE HISTORY {table_name}")
    return (
        history.select(
            "version", "timestamp", "operation",
            F.col("operationMetrics").getItem("numOutputRows").cast("int").alias("rows_written"),
            F.col("operationMetrics").getItem("numTargetRowsInserted").cast("int").alias("rows_inserted"),
            F.col("operationMetrics").getItem("numTargetRowsUpdated").cast("int").alias("rows_updated"),
        )
        .orderBy("version")
        .toPandas()
    )

ops_pd = get_delta_operations_summary(spark, "main.default.phase4")

fig = go.Figure()
fig.add_bar(x=ops_pd["version"].astype(str), y=ops_pd["rows_inserted"], name="Insérées")
fig.add_bar(x=ops_pd["version"].astype(str), y=ops_pd["rows_updated"], name="Mises à jour")
fig.update_layout(barmode="stack", title="Lignes touchées par opération, par version Delta")
fig.show()

**Partitionné vs non-partitionné**

Trois stratégies comparées (même filtre sur gold flat vs table Delta partitionnée) :

1. `Country` — filtre géo simple (`France`)
2. `year_month` — filtre temporel (`2011-11`, pic de CA)
3. `Continent` + `Country` — pruning multi-niveaux (`Europa` + `France`)

In [0]:
from src.analytics.performance_analysis import (
    ensure_partitioned_gold_variants,
    benchmark_partition_strategies,
)

In [0]:
# Création des 3 variantes partitionnées
partitioned_paths = ensure_partitioned_gold_variants(spark, gold)
print("Tables partitionnées :", partitioned_paths)

# Benchmark : flat vs partitionné pour chaque stratégie
bench_rows = benchmark_partition_strategies(
    spark,
    gold,
    partitioned_paths,
    country="France",
    year_month="2011-11",
    continent="Europa",
)
display(spark.createDataFrame(bench_rows).select(
    "strategy", "flat_seconds", "partitioned_seconds", "speedup_pct"
))

**Lecture du benchmark partitionnement**

- Un **speedup_pct > 0** signifie que la table partitionnée est plus rapide (partition pruning).
- `year_month` est souvent le plus parlant car les analyses temporelles filtrant un mois ne lisent qu’un dossier.
- `Continent + Country` illustre le pruning hiérarchique (Europa puis France).
- Sur un petit volume (~360k lignes), les gains peuvent être modestes ou variables (cache, serverless), mais l’intérêt pédagogique reste la structure des partitions (`Explain` / listing des dossiers). Sur ce volume, le gain de pruning est probablement inférieur à l'overhead de fragmentation, mais la structure démontre le principe pour un contexte à plus grande échelle.

In [0]:
strategies = [r["strategy"] for r in bench_rows]
flat_times = [r["flat_seconds"] for r in bench_rows]
part_times = [r["partitioned_seconds"] for r in bench_rows]

fig = go.Figure(
    data=[
        go.Bar(name="Non partitionné", x=strategies, y=flat_times, text=[f"{v:.3f}s" for v in flat_times], textposition="auto"),
        go.Bar(name="Partitionné", x=strategies, y=part_times, text=[f"{v:.3f}s" for v in part_times], textposition="auto"),
    ]
)
fig.update_layout(
    barmode="group",
    title="Benchmark partitionnement : 3 stratégies (filtre ciblé)",
    yaxis_title="Secondes",
    xaxis_title="Stratégie",
)
fig.show()

In [0]:
# Aperçu des partitions (dossiers / valeurs distinctes)
print("=== year_month (valeurs de partition) ===")
(
    spark.read.format("delta")
    .load(partitioned_paths["by_year_month"])
    .select("year_month")
    .distinct()
    .orderBy("year_month")
    .show(20, truncate=False)
)

print("=== Continent × Country (extrait) ===")
(
    spark.read.format("delta")
    .load(partitioned_paths["by_continent_country"])
    .select("Continent", "Country")
    .distinct()
    .orderBy("Continent", "Country")
    .show(20, truncate=False)
)

# Dossiers physiques sous le volume
print("=== Dossiers year_month ===")
display(dbutils.fs.ls(partitioned_paths["by_year_month"]))

In [0]:
from src.analytics.performance_analysis import get_query_plan, compare_query_plans

In [0]:
# Sur une vue temporaire / table UC
spark.read.format("delta").load(gold).createOrReplaceTempView("gold_flat")
spark.read.format("delta").load(partitioned_paths["by_country"]).createOrReplaceTempView("gold_by_country")

print(get_query_plan(
    spark,
    "SELECT COUNT(*) FROM gold_by_country WHERE Country = 'France'",
    extended=True,
))

plans = compare_query_plans(
    spark,
    "SELECT COUNT(*) FROM gold_flat WHERE Country = 'France'",
    "SELECT COUNT(*) FROM gold_by_country WHERE Country = 'France'",
    extended=True,
)
print(plans["plan_a"])
print(plans["plan_b"])

**Conclusion :**

La stratégie à adopter au vu de ce que nous montre nos données, est qu'il faut cibler et prioriser le top 20% des clients car ils sont responsable à eux seul de 80% du CA.

Certains articles se distingue en revenue ou en volume, c'est sur ces articles qu'il faut insister lors des campagnes de promotions.

Les différentes étapes de nettoyage des données de la phase bronze à silver ont permis de supprimés près d'un tiers des enregistrements (181306 lignes) car elles étaient malformées ou inexploitables.

La consolidation de nos données avec la fusion de notre échantillon et le reste de la base de données nous permet de confirnmer que la standardisation de nos étapes de nétoyyage est bonnecar aucune ligne n'a eu a être mise à jour.

La mise en place de différentes stratégies de partitionnement nous permet de voir l'intérêt de celui-ci au travers l'amélioration du temps d'exécution de nos requêtes, bien que ces améliorations sont à mremettre dans un contexte qui est que nous avons une base de données assez petite, sur des volumes plus importants l'intérêt serait d'autant plus grand.